In [1]:
import pandas as pd
from sqlalchemy import select
import matplotlib.pyplot as plt
import seaborn as sns

from app.config import settings
from app.db import get_session
from app.models import Parcel, NearbyParcelIdsParams
from app.nearby_parcel_request import fetch_nearby_ids, fetch_pids

pd.set_option("display.max_columns", None)

params = NearbyParcelIdsParams(
    lon=-79.9513103495131,
    lat=32.706641645949055,
    distance=5.0,
)

nearby_objectids_response = fetch_nearby_ids(params)

assert nearby_objectids_response is not None

nearby_pids = fetch_pids(nearby_objectids_response.objectIds)
len(nearby_pids), nearby_pids[:10]

(25796,
 ['4281100063',
  '3150000515',
  '3340500042',
  '4570803115',
  '4261600134',
  '4211100262',
  '4571602005',
  '4260700126',
  '3150000348',
  '3400300087'])

In [5]:
session = next(get_session())

stmt = select(Parcel).where(
    Parcel.pid.in_(nearby_pids)
)

nearby_df = pd.read_sql_query(stmt, con=session.bind)
nearby_over_1 = nearby_df[nearby_df["deeded_acreage"] > 1.0]
nearby_over_1.shape
nearby_df.shape

(25792, 32)

In [8]:
sample = nearby_df[
    nearby_df["deeded_acreage"].notna()
    & (nearby_df["deeded_acreage"] > 0)
].sample(50, random_state=42)

In [10]:
objectid = sample["objectid"]
pid = sample["pid"]
deeded_acreage = sample["deeded_acreage"]

objectid, pid, deeded_acreage

(14289    108594
 6399      48562
 25585    195792
 16202    123684
 17091    130667
 13493    102598
 8817      67142
 1757      13823
 653        5199
 8414      63908
 16761    127958
 21045    160832
 946        7428
 20919    159980
 12549     95507
 13986    106425
 2919      22398
 3559      27338
 3263      25099
 951        7445
 14880    113388
 18926    144879
 2037      16015
 17259    132118
 24747    189508
 23051    176503
 4747      36389
 17026    130173
 8908      67797
 7416      56122
 7885      59695
 4314      33126
 10958     83499
 23603    180773
 11883     90485
 4510      34661
 3147      24248
 17700    135625
 16748    127877
 22267    170704
 25548    195537
 22241    170519
 18090    138596
 18036    138190
 24912    190777
 17273    132262
 14679    111871
 6911      52500
 17333    132717
 3969      30497
 Name: objectid, dtype: int64,
 14289    4250400130
 6399     4280000040
 25585    3400300083
 16202    3340400038
 17091    3150000477
 13493    3370

In [16]:
import requests

PARCEL_LAYER_URL = "https://gisccapps.charlestoncounty.org/arcgis/rest/services/GIS_VIEWER/New_Parcel_Search/MapServer/61/query"

params = {
    "f": "json",
    "where": f"OBJECTID={','.join(objectid)}"
    "outFields": "OBJECTID,PID,ACREAGE",
    "returnGeometry": "true",
}

response = requests.get(PARCEL_LAYER_URL, params=params)
response.raise_for_status()

data = response.json()
data.keys()
data['error']
# feature = data["features"][0]
# geometry = feature["geometry"]

{'code': 400,
 'extendedCode': -2147220985,
 'message': 'Unable to complete operation.',
 'details': []}

In [21]:
f"OBJECTID={','.join(objectid)}"

TypeError: sequence item 0: expected str instance, int found

In [22]:
sample

,pid,objectid,snapshot_at,owner1,owner2,tax_district,class_code,mail_st_no,mail_st_name,mail_st_type,mail_2nd_addr,mail_2nd_addt,mail_city,mail_state,mail_zip,mail_country,legal_descr,subdivision,deeded_acreage,legal_residence,other,agr,deed_book_page,plat_book_page,sale_price,recorded_date,doc_date,geometry_esri_json,geometry_wkid,computed_area_sqft,computed_acreage,area_computation_method
14289,4250400130,108594,2026-06-19 19:23:10.606020+00:00,SKLADZINSKI ERIK HUUS,NaN,3-8,101 - RESID-SFR,1441,BROOKBANK,AVE,NaN,NaN,CHARLESTON,SC,29412,NaN,LT 82 A,NaN,0.18,Y,N,N,0672-906,N-80,289000.0,2017-10-13,2017-10-04,None,None,None,None,None
6399,4280000040,48562,2026-06-19 19:23:10.606020+00:00,CITY OF CHARLESTON,NaN,3-5,990 - UNDEVELOPABLE,NaN,NaN,NaN,PO BOX,304,CHARLESTON,SC,29402-0304,NaN,TRACT A OF PART D,NaN,26.70,N,N,N,K296-204,EA-96,1.0,1998-01-27,1998-01-23,None,None,None,None,None
25585,3400300083,195792,2026-06-19 19:23:10.606020+00:00,STEPHENS JUDITH M,NaN,3-4,101 - RESID-SFR,2004,FLEMING WOODS RD,NaN,NaN,NaN,CHARLESTON,SC,29412,NaN,Block Lot 45 GovLot Tract,FLEMING PARK,0.08,Y,N,N,0663-821,L16- 0334-0338,382035.0,2017-09-01,2017-08-31,None,None,None,None,None
16202,3340400038,123684,2026-06-19 19:23:10.606020+00:00,GOURDINE ALFRED W LIFE ESTATE,NaN,3-5,101 - RESID-SFR,1940,N GRIMBALL,RD,NaN,NaN,CHARLESTON,SC,29412,NaN,Lot 1,JAMES ISLAND,0.50,Y,N,N,1051-202,L13- 0415,10.0,2021-11-10,2021-11-08,None,None,None,None,None
17091,3150000477,130667,2026-06-19 19:23:10.606020+00:00,MCCRACKEN PARKING LLC 1,NaN,5-2,101 - RESID-SFR,10400,NE 4TH ST SUITE 2225,NaN,NaN,NaN,BELLEVUE,WA,98004,NaN,Block Lot 324 GovLot Tract,STONOVIEW PHASE 3,0.17,Y,N,N,1368-612,L17- 0478,5.0,2026-02-11,2026-02-03,None,None,None,None,None
13493,3370000167,102598,2026-06-19 19:23:10.606020+00:00,CHARLESTON ISLANDS III LLC,NaN,3-4,990 - UNDEVELOPABLE,397,LITTLE NECK,RD,NaN,NaN,VIRGINIA BEACH,VA,23452-5765,NaN,TRACT R,NaN,0.82,N,N,N,B478-112,ED-722,9.0,2003-12-09,2003-12-01,None,None,None,None,None
8817,3370600027,67142,2026-06-19 19:23:10.606020+00:00,JAKUBEK TOMAS,JUILLARD MARIETTA,3-4,120 - RESID-TWH,63,MAPLE,ST,NaN,NaN,CHARLESTON,SC,29403,NaN,LOT 27 PHASE I,NaN,0.09,N,N,N,0412-299,EE-492,140000.0,2014-06-20,2014-06-18,None,None,None,None,None
1757,4281600085,13823,2026-06-19 19:23:10.606020+00:00,HESS BENJAMIN A,HESS STEPHANIE S,3-4,101 - RESID-SFR,551,CECILIA COVE,DR,NaN,NaN,CHARLESTON,SC,29412,NaN,LOT 65,NaN,0.22,Y,N,N,0518-155,EE-783,300000.0,2015-11-18,2015-11-16,None,None,None,None,None
653,3301100070,5199,2026-06-19 19:23:10.606020+00:00,ALBAUGH FORREST J,ALBAUGH ASHLEY,3-1,101 - RESID-SFR,2198,BROWN PELICAN,LN,NaN,NaN,CHARLESTON,SC,29412,NaN,LOT 6,NaN,0.38,Y,N,N,1299-156,EJ-537,5.0,2025-03-10,2025-03-03,None,None,None,None,None
8414,4261100246,63908,2026-06-19 19:23:10.606020+00:00,CZARNIECKI ROBERT E,CZARNIECKI JOAN A,3-4,101 - RESID-SFR,575,WHITE CHAPEL,CIR,NaN,NaN,CHARLESTON,SC,29412-4349,NaN,LOT 21,NaN,0.26,Y,N,N,0415-947,EE-738,10.0,2014-07-10,2014-06-30,None,None,None,None,None
